# News Crawler v2 — Unified Schema (JSONL output)

Notebook này crawl tin tức từ 14 báo VN và lưu ra **JSONL** theo schema thống nhất,
sẵn sàng import vào **Elasticsearch + ClickHouse**.

**Cấu trúc:**
1. **Shared utilities** — code dùng chung cho cả 4 cell (HTTP session, storage, utils...)
2. **RSS crawler** (`tong hop`) — 100+ RSS feeds từ nhiều báo, có enrich full content
3. **Lao Động crawler** — HTML category pages (có xử lý cookie challenge)
4. **ZNews crawler** — HTML category pages
5. **24h crawler** — RSS của 24h.com.vn

**Schema output (mỗi dòng JSONL):**
```json
{
  "id": "md5(url)",
  "url": "...",
  "title": "...",
  "content": "...",           // đã strip boilerplate
  "published_at": "ISO UTC",
  "crawled_at": "ISO UTC",
  "source": "vnexpress",      // tên báo chuẩn hóa (parse từ domain)
  "source_domain": "vnexpress.net",
  "source_type": "rss" | "html",
  "category_raw": "Thế giới",
  "category_normalized": "world",
  "author": "Đức Hoàng",      // tách khỏi source (bug cũ: source.name bị nhầm author)
  "language": "vi",
  "keywords": ["..."],
  "entities": [],             // để trống, populate bằng NER bước sau
  "content_length": 3218,
  "has_full_content": true    // True nếu content >= 500 chars
}
```

**Chạy:** Luôn chạy Cell 1 (shared utilities) trước. Sau đó có thể chạy Cell 2/3/4/5 theo thứ tự bất kỳ.

## Cell 1 — Shared utilities (chạy đầu tiên)

In [ ]:
# -*- coding: utf-8 -*-
"""
SHARED UTILITIES — chạy cell này TRƯỚC các cell khác.
Định nghĩa: HTTP client, JSONL storage, dedup, utils chuẩn hóa
            date/URL/category/content, Article dataclass, extractors.
"""

import os
import re
import json
import time
import random
import hashlib
import logging
from dataclasses import dataclass, asdict, field
from datetime import datetime, timezone, timedelta
from collections import defaultdict
from typing import Any, Dict, List, Optional, Set, Tuple
from urllib.parse import urlparse, urljoin

import requests
import feedparser
from bs4 import BeautifulSoup
from dateutil import parser as dateparser
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

VN_TZ = timezone(timedelta(hours=7))


# ====================================================================
# Logger
# ====================================================================
def setup_logger(name, log_file=None, level=logging.INFO):
    logger = logging.getLogger(name)
    logger.setLevel(level)
    if logger.handlers:
        return logger
    fmt = logging.Formatter(
        "%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
        datefmt="%H:%M:%S",
    )
    sh = logging.StreamHandler()
    sh.setFormatter(fmt)
    logger.addHandler(sh)
    if log_file:
        os.makedirs(os.path.dirname(log_file) or ".", exist_ok=True)
        fh = logging.FileHandler(log_file, encoding="utf-8")
        fh.setFormatter(fmt)
        logger.addHandler(fh)
    return logger


log = setup_logger("crawler")


# ====================================================================
# Schema
# ====================================================================
@dataclass
class Article:
    id: str
    url: str
    title: str
    content: str
    published_at: str          # ISO-8601 UTC
    crawled_at: str            # ISO-8601 UTC
    source: str                # "vnexpress", "dantri", ...
    source_domain: str
    source_type: str           # "rss" | "html"
    category_raw: str = ""
    category_normalized: str = ""
    author: str = ""
    language: str = "vi"
    keywords: List[str] = field(default_factory=list)
    entities: List[Dict[str, str]] = field(default_factory=list)
    content_length: int = 0
    has_full_content: bool = False

    def to_dict(self):
        return asdict(self)


# ====================================================================
# URL & ID utilities
# ====================================================================
def normalize_url(url: str) -> str:
    if not url:
        return ""
    url = url.strip()
    if "#" in url:
        url = url.split("#", 1)[0]
    p = urlparse(url)
    if p.path.endswith("/") and len(p.path) > 1:
        url = url[:-1]
    return url


def stable_id(url: str) -> str:
    return hashlib.md5(normalize_url(url).encode("utf-8")).hexdigest()


def get_domain(url: str) -> str:
    try:
        return urlparse(url).netloc.lower().replace("www.", "")
    except Exception:
        return ""


DOMAIN_TO_SOURCE = {
    "vnexpress.net": "vnexpress",
    "dantri.com.vn": "dantri",
    "tuoitre.vn": "tuoitre",
    "thanhnien.vn": "thanhnien",
    "vietnamnet.vn": "vietnamnet",
    "laodong.vn": "laodong",
    "nld.com.vn": "nld",
    "vietnamplus.vn": "vietnamplus",
    "soha.vn": "soha",
    "nhandan.vn": "nhandan",
    "baotintuc.vn": "baotintuc",
    "kienthuc.net.vn": "kienthuc",
    "znews.vn": "znews",
    "24h.com.vn": "24h",
}


def domain_to_source(domain: str) -> str:
    domain = domain.lower().replace("www.", "")
    if domain in DOMAIN_TO_SOURCE:
        return DOMAIN_TO_SOURCE[domain]
    for kd, src in DOMAIN_TO_SOURCE.items():
        if domain.endswith("." + kd) or domain.endswith(kd):
            return src
    return domain


# ====================================================================
# Date utils
# ====================================================================
def to_iso_utc(raw: Optional[str]) -> Optional[str]:
    if not raw:
        return None
    raw = str(raw).strip()
    if not raw:
        return None
    try:
        dt = dateparser.parse(raw)
        if dt is None:
            return None
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=VN_TZ)
        return dt.astimezone(timezone.utc).isoformat()
    except Exception:
        return None


def iso_to_local_date(iso_utc: Optional[str]) -> Optional[str]:
    if not iso_utc:
        return None
    try:
        dt = dateparser.parse(iso_utc)
        if dt is None:
            return None
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt.astimezone(VN_TZ).date().isoformat()
    except Exception:
        return None


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


# ====================================================================
# Category normalization
# ====================================================================
CATEGORY_MAP = {
    # sports
    "the-thao": "sports", "thể thao": "sports", "bong-da": "sports", "xe": "sports",
    "sea-games-32": "sports", "asiancup2019": "sports",
    # news
    "thoi-su": "news", "thời sự": "news", "tin-tuc-trong-ngay": "news",
    "xa-hoi": "news", "xã hội": "news", "tin-moi-nhat": "news", "tin tức": "news",
    # world
    "the-gioi": "world", "thế giới": "world", "quoc-te": "world", "quốc tế": "world",
    "asean": "world", "chau-a-tbd": "world", "chau-au": "world", "chau-my": "world",
    "chau-phi": "world", "trung-dong": "world",
    # politics
    "chinh-tri": "politics", "chính trị": "politics", "xay-dung-dang": "politics",
    "xa-luan": "politics",
    # business
    "kinh-doanh": "business", "kinh-te": "business", "kinh doanh": "business",
    "kinh tế": "business", "kinh-doanh-tai-chinh": "business",
    "taichinh": "business", "chungkhoan": "business", "tai-chinh-bat-dong-san": "business",
    "bat-dong-san": "business", "thi-truong": "business",
    # entertainment
    "giai-tri": "entertainment", "giải trí": "entertainment", "van-hoa": "entertainment",
    "van-hoa-giai-tri": "entertainment", "van-hoa-van-nghe": "entertainment",
    "phim": "entertainment", "xuat-ban": "entertainment",
    # lifestyle
    "doi-song": "lifestyle", "đời sống": "lifestyle", "gia-dinh": "lifestyle",
    "ban-tre-cuoc-song": "lifestyle", "thoi-trang": "lifestyle",
    # law / education / health / tech / travel / auto / military
    "phap-luat": "law", "pháp luật": "law",
    "giao-duc": "education", "giáo dục": "education", "giao-duc-du-hoc": "education",
    "giao-duc-khoa-hoc": "education", "giaoduc": "education",
    "suc-khoe": "health", "sức khỏe": "health", "y-te": "health", "yte": "health",
    "cong-nghe": "tech", "công nghệ": "tech", "hi-tech": "tech", "so-hoa": "tech",
    "khoa-hoc": "tech", "khoa học": "tech", "ai-365": "tech",
    "du-lich": "travel", "du lịch": "travel", "du-lich-xanh": "travel",
    "oto-xe-may": "auto", "o-to-xe-may": "auto",
    "quan-su": "military", "quân sự": "military",
    "trang-chu": "home", "home": "home", "tin-moi": "home",
}


def normalize_category(raw: str) -> str:
    if not raw:
        return "other"
    key = raw.strip().lower()
    if key in CATEGORY_MAP:
        return CATEGORY_MAP[key]
    for k, v in CATEGORY_MAP.items():
        if k in key or key in k:
            return v
    return "other"


def extract_category_from_url(url: str) -> str:
    try:
        parts = [p for p in urlparse(url).path.split("/") if p]
        for part in parts[:2]:
            clean = re.sub(r"-\d+$", "", part)
            if normalize_category(clean) != "other":
                return clean
        return parts[0] if parts else ""
    except Exception:
        return ""


# ====================================================================
# Content cleaning
# ====================================================================
BOILERPLATE_PATTERNS = [
    r"^Lưu bài viết thành công.*?Tin bài đã lưu",
    r"Nguồn:\s*\[Link nguồn\].*$",
    r"^\(Dân trí\)\s*-\s*",
    r"^TTO\s*-\s*",
    r"^TTXVN\s*-\s*",
]
_BOILERPLATE_REGEXES = [re.compile(p, re.DOTALL | re.IGNORECASE) for p in BOILERPLATE_PATTERNS]


def clean_content(text: str) -> str:
    if not text:
        return ""
    t = text
    for rx in _BOILERPLATE_REGEXES:
        t = rx.sub("", t).strip()
    t = re.sub(r"\s+", " ", t).strip()
    return t


# ====================================================================
# Validation thresholds
# ====================================================================
MIN_CONTENT_LENGTH_STRICT = 500   # HTML crawler
MIN_CONTENT_LENGTH_LOOSE = 100    # RSS (có thể là summary)


def is_article_valid(title, content, published_at, min_content_len=MIN_CONTENT_LENGTH_LOOSE):
    if not title or not title.strip():
        return False, "empty_title"
    if not published_at:
        return False, "no_published_at"
    if not content or len(content) < min_content_len:
        return False, f"content_too_short({len(content) if content else 0})"
    return True, ""


# ====================================================================
# HTTP client — retry + per-domain rate limit
# ====================================================================
class PerDomainRateLimiter:
    def __init__(self, min_delay=0.5, jitter=0.3):
        self.min_delay = min_delay
        self.jitter = jitter
        self._last_call = defaultdict(float)

    def wait(self, url):
        domain = get_domain(url)
        now = time.time()
        last = self._last_call[domain]
        elapsed = now - last
        delay = self.min_delay + random.uniform(0, self.jitter)
        if elapsed < delay:
            time.sleep(delay - elapsed)
        self._last_call[domain] = time.time()


class HttpClient:
    DEFAULT_HEADERS = {
        "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                       "AppleWebKit/537.36 (KHTML, like Gecko) "
                       "Chrome/120.0.0.0 Safari/537.36"),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "vi,en;q=0.9",
    }

    def __init__(self, timeout=25, min_delay_per_domain=0.5, jitter=0.3, extra_headers=None):
        self.timeout = timeout
        self.rate_limiter = PerDomainRateLimiter(min_delay_per_domain, jitter)
        self.session = requests.Session()
        headers = dict(self.DEFAULT_HEADERS)
        if extra_headers:
            headers.update(extra_headers)
        self.session.headers.update(headers)
        retry = Retry(
            total=5, connect=5, read=5, backoff_factor=0.6,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["GET", "HEAD"],
            respect_retry_after_header=True,
            raise_on_status=False,
        )
        adapter = HTTPAdapter(max_retries=retry, pool_connections=50, pool_maxsize=50)
        self.session.mount("http://", adapter)
        self.session.mount("https://", adapter)

    def get(self, url, **kwargs):
        self.rate_limiter.wait(url)
        try:
            r = self.session.get(url, timeout=self.timeout, allow_redirects=True, **kwargs)
        except requests.RequestException as e:
            log.warning(f"GET failed {url}: {e}")
            return None
        # laodong.vn cookie challenge
        if r.status_code == 200 and "document.cookie" in r.text[:1000] and len(r.content) < 1500:
            m = re.search(r'document\.cookie\s*=\s*"([^";]+)', r.text)
            if m and "=" in m.group(1):
                name, value = m.group(1).split("=", 1)
                self.session.cookies.set(name.strip(), value.strip())
                time.sleep(0.5)
                try:
                    r = self.session.get(url, timeout=self.timeout, allow_redirects=True, **kwargs)
                except requests.RequestException:
                    return None
        if r.status_code >= 400:
            log.warning(f"HTTP {r.status_code} on {url}")
            return None
        return r

    def get_text(self, url):
        r = self.get(url)
        return r.text if r is not None else None


# ====================================================================
# JSONL storage — append-safe, resume-safe
# ====================================================================
class JsonlStore:
    def __init__(self, path):
        safe = path.replace(":", "-")
        os.makedirs(os.path.dirname(safe) or ".", exist_ok=True)
        self.path = safe
        self._seen_ids = set()
        self._seen_urls = set()
        self._load()

    def _load(self):
        if not os.path.exists(self.path):
            return
        n = 0
        with open(self.path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    o = json.loads(line)
                    if "id" in o:
                        self._seen_ids.add(o["id"])
                    if "url" in o:
                        self._seen_urls.add(o["url"])
                    n += 1
                except json.JSONDecodeError:
                    continue
        log.info(f"Loaded {n:,} existing records from {self.path}")

    def has_id(self, aid):
        return aid in self._seen_ids

    def has_url(self, url):
        return normalize_url(url) in self._seen_urls

    def append(self, article: Article) -> bool:
        if article.id in self._seen_ids:
            return False
        with open(self.path, "a", encoding="utf-8") as f:
            f.write(json.dumps(article.to_dict(), ensure_ascii=False) + "\n")
            f.flush()
        self._seen_ids.add(article.id)
        self._seen_urls.add(article.url)
        return True

    def count(self):
        return len(self._seen_ids)


# ====================================================================
# HTML extractors (JSON-LD + meta + site-specific selectors)
# ====================================================================
SITE_BODY_SELECTORS = {
    "vnexpress.net":   [".fck_detail", "article.fck_detail"],
    "dantri.com.vn":   [".singular-content", ".dt-news__content", ".detail-content"],
    "tuoitre.vn":      [".detail-content", ".content-detail", "article.detail"],
    "thanhnien.vn":    [".detail-cmain", ".detail__content"],
    "vietnamnet.vn":   [".maincontent", ".content-detail"],
    "laodong.vn":      ["article.detail-content", ".detail-content"],
    "nld.com.vn":      [".detail-content", "article .detail-cmain"],
    "vietnamplus.vn":  [".article-body", ".article__body", ".the-article-body"],
    "soha.vn":         [".detail-content-body", ".news-content"],
    "nhandan.vn":      [".article__body", ".detail-content"],
    "baotintuc.vn":    [".detail-content", ".article-body"],
    "kienthuc.net.vn": [".article-body", ".the-article-body"],
    "znews.vn":        [".the-article-body", "article .article-content"],
    "24h.com.vn":      [".text-conent", ".cate-24h-foot-home-detail-cmain", ".detail-news-content"],
}

NOISE_SELECTORS = [
    "script", "style", "noscript", "iframe",
    ".ads", ".ad-container", ".advertisement",
    ".related-news", ".related-articles", ".box-tinlienquan",
    ".social-share", ".share-bar", ".share-box",
    ".comment", ".comments", "#comments",
    ".author-box", ".author-info",
    ".newsletter", ".subscribe",
    "figcaption",
]


def extract_jsonld(soup: BeautifulSoup) -> Dict[str, Any]:
    out = {}
    for tag in soup.find_all("script", attrs={"type": "application/ld+json"}):
        txt = tag.string or tag.get_text() or ""
        if not txt.strip():
            continue
        try:
            data = json.loads(txt)
        except json.JSONDecodeError:
            try:
                data = json.loads(re.sub(r",\s*([}\]])", r"\1", txt))
            except Exception:
                continue
        candidates = data if isinstance(data, list) else [data]
        for item in candidates:
            if not isinstance(item, dict):
                continue
            t = item.get("@type", "")
            if isinstance(t, list):
                t = " ".join(t)
            if "Article" not in t and "NewsArticle" not in t:
                continue
            if "headline" in item and "title" not in out:
                out["title"] = str(item["headline"]).strip()
            if "datePublished" in item and "published_at" not in out:
                out["published_at"] = to_iso_utc(item["datePublished"]) or ""
            if "author" in item and "author" not in out:
                a = item["author"]
                if isinstance(a, dict):
                    out["author"] = a.get("name", "")
                elif isinstance(a, list) and a:
                    first = a[0]
                    out["author"] = first.get("name", "") if isinstance(first, dict) else str(first)
                else:
                    out["author"] = str(a)
            if "articleSection" in item and "category_raw" not in out:
                sec = item["articleSection"]
                if isinstance(sec, list):
                    sec = sec[0] if sec else ""
                out["category_raw"] = str(sec)
            if "keywords" in item and "keywords" not in out:
                kw = item["keywords"]
                if isinstance(kw, list):
                    out["keywords"] = [str(k).strip() for k in kw if k]
                elif isinstance(kw, str):
                    out["keywords"] = [k.strip() for k in kw.split(",") if k.strip()]
            if "articleBody" in item and "content" not in out:
                out["content"] = str(item["articleBody"])
    return out


def extract_meta(soup: BeautifulSoup) -> Dict[str, Any]:
    out = {}
    og = soup.select_one('meta[property="og:title"]')
    if og and og.get("content"):
        out["title"] = og["content"].strip()
    elif soup.title:
        out["title"] = soup.title.get_text(strip=True)
    for sel in ['meta[property="article:published_time"]',
                'meta[itemprop="datePublished"]',
                'meta[name="pubdate"]']:
        t = soup.select_one(sel)
        if t and t.get("content"):
            pub = to_iso_utc(t["content"].strip())
            if pub:
                out["published_at"] = pub
                break
    if "published_at" not in out:
        t = soup.select_one("time[datetime]")
        if t and t.get("datetime"):
            pub = to_iso_utc(t["datetime"])
            if pub:
                out["published_at"] = pub
    sec = soup.select_one('meta[property="article:section"]')
    if sec and sec.get("content"):
        out["category_raw"] = sec["content"].strip()
    kw = soup.select_one('meta[name="keywords"]')
    if kw and kw.get("content"):
        out["keywords"] = [k.strip() for k in kw["content"].split(",") if k.strip()]
    au = soup.select_one('meta[name="author"]') or soup.select_one('meta[property="article:author"]')
    if au and au.get("content"):
        out["author"] = au["content"].strip()
    return out


def _extract_body(soup: BeautifulSoup, selectors: List[str]) -> str:
    for sel in selectors:
        body = soup.select_one(sel)
        if not body:
            continue
        copy = BeautifulSoup(str(body), "lxml")
        for ns in NOISE_SELECTORS:
            for tag in copy.select(ns):
                tag.decompose()
        paragraphs = copy.find_all("p")
        if paragraphs:
            parts = [p.get_text(" ", strip=True) for p in paragraphs]
            parts = [p for p in parts if p and len(p) > 10]
            if parts:
                return " ".join(parts)
        return copy.get_text(" ", strip=True)
    return ""


def extract_article(html: str, url: str) -> Dict[str, Any]:
    if not html:
        return {}
    soup = BeautifulSoup(html, "lxml")
    domain = urlparse(url).netloc.lower().replace("www.", "")
    out = extract_jsonld(soup)
    for k, v in extract_meta(soup).items():
        if not out.get(k):
            out[k] = v
    if not out.get("content"):
        selectors = None
        for kd, sels in SITE_BODY_SELECTORS.items():
            if kd in domain:
                selectors = sels
                break
        if selectors:
            body = _extract_body(soup, selectors)
            if body:
                out["content"] = body
    if "content" in out:
        out["content"] = clean_content(out["content"])
    if not out.get("category_raw"):
        cat = extract_category_from_url(url)
        if cat:
            out["category_raw"] = cat
    kw = out.get("keywords", [])
    if isinstance(kw, str):
        kw = [k.strip() for k in kw.split(",") if k.strip()]
    out["keywords"] = [str(k).strip() for k in kw if k]
    return out


# ====================================================================
# Article builder
# ====================================================================
def build_article(url, title, content, pub_iso, source_type,
                  category_raw="", author="", keywords=None) -> Article:
    """Helper để build Article object, tự tính các field derived."""
    url = normalize_url(url)
    domain = get_domain(url)
    content = clean_content(content) if content else ""
    return Article(
        id=stable_id(url),
        url=url,
        title=(title or "").strip(),
        content=content,
        published_at=pub_iso or "",
        crawled_at=utc_now_iso(),
        source=domain_to_source(domain),
        source_domain=domain,
        source_type=source_type,
        category_raw=category_raw,
        category_normalized=normalize_category(category_raw),
        author=author or "",
        language="vi",
        keywords=keywords or [],
        entities=[],
        content_length=len(content),
        has_full_content=len(content) >= 500,
    )


print("✓ Shared utilities loaded")
print("  - Article schema:", list(Article.__dataclass_fields__.keys()))
print("  - Domains known  :", len(DOMAIN_TO_SOURCE))
print("  - Category map   :", len(set(CATEGORY_MAP.values())), "taxonomies")


## Cell 2 — RSS crawler (tong hop) — 100+ feeds với enrich full content

In [ ]:
# ====================================================================
# CELL 2 — RSS CRAWLER (tong hop)
# Crawl tất cả RSS feeds, enrich full content từ URL gốc
# ====================================================================

# -------- CONFIG --------
RSS_FEEDS = [
    # VnExpress
    "https://vnexpress.net/rss/tin-moi-nhat.rss",
    "https://vnexpress.net/rss/thoi-su.rss",
    "https://vnexpress.net/rss/the-gioi.rss",
    "https://vnexpress.net/rss/kinh-doanh.rss",
    "https://vnexpress.net/rss/giai-tri.rss",
    "https://vnexpress.net/rss/the-thao.rss",
    "https://vnexpress.net/rss/phap-luat.rss",
    "https://vnexpress.net/rss/giao-duc.rss",
    "https://vnexpress.net/rss/suc-khoe.rss",
    "https://vnexpress.net/rss/gia-dinh.rss",
    "https://vnexpress.net/rss/du-lich.rss",
    "https://vnexpress.net/rss/khoa-hoc.rss",
    "https://vnexpress.net/rss/so-hoa.rss",
    "https://vnexpress.net/rss/oto-xe-may.rss",
    "https://vnexpress.net/rss/y-kien.rss",
    # Dân Trí
    "https://dantri.com.vn/rss/trang-chu.rss",
    "https://dantri.com.vn/rss/xa-hoi.rss",
    "https://dantri.com.vn/rss/the-gioi.rss",
    "https://dantri.com.vn/rss/kinh-doanh.rss",
    "https://dantri.com.vn/rss/the-thao.rss",
    "https://dantri.com.vn/rss/giai-tri.rss",
    "https://dantri.com.vn/rss/giao-duc.rss",
    "https://dantri.com.vn/rss/suc-khoe.rss",
    "https://dantri.com.vn/rss/du-lich.rss",
    "https://dantri.com.vn/rss/o-to-xe-may.rss",
    # Tuổi Trẻ
    "https://tuoitre.vn/rss/tin-moi-nhat.rss",
    "https://tuoitre.vn/rss/thoi-su.rss",
    "https://tuoitre.vn/rss/the-gioi.rss",
    "https://tuoitre.vn/rss/phap-luat.rss",
    "https://tuoitre.vn/rss/kinh-doanh.rss",
    "https://tuoitre.vn/rss/giao-duc.rss",
    "https://tuoitre.vn/rss/the-thao.rss",
    "https://tuoitre.vn/rss/giai-tri.rss",
    "https://tuoitre.vn/rss/xe.rss",
    # Thanh Niên
    "https://thanhnien.vn/rss/home.rss",
    "https://thanhnien.vn/rss/thoi-su.rss",
    "https://thanhnien.vn/rss/the-gioi.rss",
    "https://thanhnien.vn/rss/kinh-te.rss",
    "https://thanhnien.vn/rss/van-hoa.rss",
    "https://thanhnien.vn/rss/the-thao.rss",
    "https://thanhnien.vn/rss/cong-nghe.rss",
    "https://thanhnien.vn/rss/gioi-tre.rss",
    # VietnamNet
    "https://vietnamnet.vn/rss/thoi-su.rss",
    "https://vietnamnet.vn/rss/the-gioi.rss",
    "https://vietnamnet.vn/rss/kinh-doanh.rss",
    "https://vietnamnet.vn/rss/giao-duc.rss",
    "https://vietnamnet.vn/rss/the-thao.rss",
    # # Lao Động
    # "https://laodong.vn/rss/home.rss",
    # "https://laodong.vn/rss/xa-hoi.rss",
    # "https://laodong.vn/rss/kinh-doanh.rss",
    # "https://laodong.vn/rss/van-hoa-giai-tri.rss",
    # "https://laodong.vn/rss/thoi-su.rss",
    # "https://laodong.vn/rss/the-gioi.rss",
    # "https://laodong.vn/rss/phap-luat.rss",
    # "https://laodong.vn/rss/the-thao.rss",
    # "https://laodong.vn/rss/suc-khoe.rss",
    # Người Lao Động
    "https://nld.com.vn/rss/home.rss",
    "https://nld.com.vn/rss/thoi-su.rss",
    "https://nld.com.vn/rss/quoc-te.rss",
    "https://nld.com.vn/rss/kinh-te.rss",
    "https://nld.com.vn/rss/suc-khoe.rss",
    "https://nld.com.vn/rss/giao-duc-khoa-hoc.rss",
    "https://nld.com.vn/rss/phap-luat.rss",
    "https://nld.com.vn/rss/giai-tri.rss",
    "https://nld.com.vn/rss/the-thao.rss",
    # VietnamPlus
    "https://www.vietnamplus.vn/rss/home.rss",
    "https://www.vietnamplus.vn/rss/chinhtri-291.rss",
    "https://www.vietnamplus.vn/rss/thegioi-209.rss",
    "https://www.vietnamplus.vn/rss/kinhte-311.rss",
    "https://www.vietnamplus.vn/rss/xahoi-314.rss",
    "https://www.vietnamplus.vn/rss/doisong-320.rss",
    "https://www.vietnamplus.vn/rss/thethao-214.rss",
    # Soha
    "https://soha.vn/rss/home.rss",
    "https://soha.vn/rss/thoi-su-xa-hoi.rss",
    "https://soha.vn/rss/kinh-doanh.rss",
    "https://soha.vn/rss/quoc-te.rss",
    "https://soha.vn/rss/the-thao.rss",
    "https://soha.vn/rss/giai-tri.rss",
    "https://soha.vn/rss/phap-luat.rss",
    # Nhân Dân
    "https://nhandan.vn/rss/home.rss",
    "https://nhandan.vn/rss/chinhtri-1171.rss",
    "https://nhandan.vn/rss/kinhte-1185.rss",
    "https://nhandan.vn/rss/phapluat-1287.rss",
    "https://nhandan.vn/rss/du-lich-1257.rss",
    "https://nhandan.vn/rss/thegioi-1231.rss",
    "https://nhandan.vn/rss/thethao-1224.rss",
    # Báo Tin Tức
    "https://baotintuc.vn/tin-moi-nhat.rss",
    "https://baotintuc.vn/thoi-su.rss",
    "https://baotintuc.vn/the-gioi.rss",
    "https://baotintuc.vn/kinh-te.rss",
    "https://baotintuc.vn/xa-hoi.rss",
    "https://baotintuc.vn/phap-luat.rss",
    "https://baotintuc.vn/giao-duc.rss",
    "https://baotintuc.vn/van-hoa.rss",
    "https://baotintuc.vn/the-thao.rss",
    "https://baotintuc.vn/quan-su.rss",
    # Kiến Thức
    "https://kienthuc.net.vn/rss/home.rss",
    "https://kienthuc.net.vn/rss/chinh-tri-348.rss",
    "https://kienthuc.net.vn/rss/xa-hoi-349.rss",
    "https://kienthuc.net.vn/rss/the-gioi-350.rss",
    "https://kienthuc.net.vn/rss/quan-su-359.rss",
    "https://kienthuc.net.vn/rss/giai-tri-365.rss",
]

START_DATE = "2026-01-18"   # chỉ lấy bài >= ngày này (theo giờ VN)
ENRICH_FULL_CONTENT = True  # True = fetch full content từ URL gốc (chậm nhưng đầy đủ)
JSONL_PATH = "data/rss_articles_old.jsonl"

# -------- HELPERS --------
def _parse_entry_date(entry):
    if getattr(entry, "published_parsed", None):
        try:
            dt = datetime(*entry.published_parsed[:6], tzinfo=timezone.utc)
            return dt.isoformat()
        except Exception:
            pass
    for attr in ("published", "updated", "pubDate"):
        val = entry.get(attr) if hasattr(entry, "get") else None
        if val:
            r = to_iso_utc(val)
            if r:
                return r
    return None


def _extract_rss_summary(entry):
    raw = ""
    if getattr(entry, "content", None):
        c = entry.content
        if isinstance(c, list) and c:
            raw = c[0].get("value", "")
    if not raw:
        raw = entry.get("summary", "") or entry.get("description", "")
    if not raw:
        return ""
    try:
        soup = BeautifulSoup(raw, "lxml")
        return clean_content(soup.get_text(" ", strip=True))
    except Exception:
        return clean_content(raw)


def _extract_rss_keywords(entry):
    kw = []
    if getattr(entry, "tags", None):
        for tag in entry.tags:
            term = tag.get("term") if hasattr(tag, "get") else None
            if term:
                term = term.strip()
                if term and term not in kw:
                    kw.append(term)
    return kw


def build_article_from_rss(entry, http, enrich=True):
    url = normalize_url(entry.get("link", ""))
    if not url:
        return None
    title = (entry.get("title") or "").strip()
    pub = _parse_entry_date(entry)
    if not pub:
        return None
    rss_kw = _extract_rss_keywords(entry)
    rss_summary = _extract_rss_summary(entry)
    content = ""
    category_raw = ""
    author = ""
    ld_kw = []
    if enrich:
        html = http.get_text(url)
        if html:
            ex = extract_article(html, url)
            content = ex.get("content", "")
            category_raw = ex.get("category_raw", "")
            author = ex.get("author", "")
            ld_kw = ex.get("keywords", [])
            if ex.get("title") and len(ex["title"]) > len(title):
                title = ex["title"]
    if not content or len(content) < MIN_CONTENT_LENGTH_LOOSE:
        if rss_summary:
            content = rss_summary
    all_kw = list(dict.fromkeys(rss_kw + ld_kw))
    if not category_raw:
        category_raw = extract_category_from_url(url)
    valid, reason = is_article_valid(title, content, pub, MIN_CONTENT_LENGTH_LOOSE)
    if not valid:
        return None
    return build_article(url, title, content, pub, "rss",
                         category_raw=category_raw, author=author, keywords=all_kw)


# -------- MAIN --------
http = HttpClient(min_delay_per_domain=0.8)
store = JsonlStore(JSONL_PATH)
start_iso = to_iso_utc(START_DATE + "T00:00:00")
start_local = iso_to_local_date(start_iso)

total = {"fetched": 0, "added": 0, "dup": 0, "old": 0, "invalid": 0}

for i, feed_url in enumerate(RSS_FEEDS, 1):
    log.info(f"[{i}/{len(RSS_FEEDS)}] Fetching feed: {feed_url}")
    feed = feedparser.parse(feed_url)
    if not feed.entries:
        log.warning(f"  No entries in {feed_url}")
        continue
    log.info(f"  {len(feed.entries)} entries")
    feed_stats = {"added": 0, "dup": 0, "old": 0, "invalid": 0}
    for entry in feed.entries:
        total["fetched"] += 1
        url = normalize_url(entry.get("link", ""))
        if not url:
            continue
        if store.has_url(url):
            total["dup"] += 1
            feed_stats["dup"] += 1
            continue
        pub_iso = _parse_entry_date(entry)
        if pub_iso and start_local:
            pub_local = iso_to_local_date(pub_iso)
            if pub_local and pub_local < start_local:
                total["old"] += 1
                feed_stats["old"] += 1
                continue
        article = build_article_from_rss(entry, http, enrich=ENRICH_FULL_CONTENT)
        if article is None:
            total["invalid"] += 1
            feed_stats["invalid"] += 1
            continue
        if store.append(article):
            total["added"] += 1
            feed_stats["added"] += 1
        else:
            total["dup"] += 1
            feed_stats["dup"] += 1
    log.info(f"  → feed result: {feed_stats}")

log.info("=" * 60)
log.info(f"FINAL: {total}")
log.info(f"Total in store: {store.count():,}")
log.info(f"Output: {JSONL_PATH}")
log.info("=" * 60)


## Cell 3 — Lao Động HTML crawler

In [ ]:
# ====================================================================
# CELL 3 — LAO ĐỘNG HTML CRAWLER
# Crawl category pages với pagination (có xử lý cookie challenge)
# ====================================================================

LAODONG_CATEGORIES = [
    "https://laodong.vn/thoi-su/",
    "https://laodong.vn/the-gioi/",
    "https://laodong.vn/xa-hoi/",
    "https://laodong.vn/phap-luat/",
    "https://laodong.vn/kinh-doanh/",
    "https://laodong.vn/bat-dong-san/",
    "https://laodong.vn/van-hoa/",
    "https://laodong.vn/giao-duc/",
    "https://laodong.vn/the-thao/",
    "https://laodong.vn/suc-khoe/",
    "https://laodong.vn/cong-nghe/",
    "https://laodong.vn/xe/",
    "https://laodong.vn/du-lich/",
]

START_DATE = "2026-01-18"
MAX_PAGES_PER_CATEGORY = 50
JSONL_PATH = "data/laodong_articles.jsonl"

LAODONG_ARTICLE_REGEX = re.compile(r"/[\w-]+/[\w-]+-\d+\.ldo")


def _laodong_page_url(category_url, page):
    if page == 1:
        return category_url
    base = category_url.rstrip("/")
    return f"{base}/trang-{page}.ldo"


def _extract_article_urls(html, base_url, url_regex):
    if not html:
        return []
    soup = BeautifulSoup(html, "lxml")
    urls = set()
    domain = get_domain(base_url)
    for a in soup.select("a[href]"):
        href = a.get("href", "").strip()
        if not href:
            continue
        full = urljoin(base_url, href)
        if get_domain(full) != domain:
            continue
        if url_regex.search(full):
            urls.add(normalize_url(full))
    return list(urls)


def _process_article_url(url, http, store, min_content_len=MIN_CONTENT_LENGTH_STRICT):
    if store.has_url(url):
        return "dup"
    html = http.get_text(url)
    if not html:
        return "fetch_failed"
    data = extract_article(html, url)
    title = data.get("title", "")
    content = data.get("content", "")
    pub = data.get("published_at", "")
    valid, reason = is_article_valid(title, content, pub, min_content_len)
    if not valid:
        return f"invalid:{reason}"
    article = build_article(
        url, title, content, pub, "html",
        category_raw=data.get("category_raw", ""),
        author=data.get("author", ""),
        keywords=data.get("keywords", []),
    )
    return "added" if store.append(article) else "dup"


# -------- MAIN --------
http = HttpClient(min_delay_per_domain=0.8)
store = JsonlStore(JSONL_PATH)
start_local = iso_to_local_date(to_iso_utc(START_DATE + "T00:00:00"))

total = {"added": 0, "dup": 0, "invalid": 0, "fetch_failed": 0, "pages": 0}

for cat_url in LAODONG_CATEGORIES:
    log.info(f">>> Category: {cat_url}")
    consecutive_empty = 0
    seen_on_cat = set()
    for page in range(1, MAX_PAGES_PER_CATEGORY + 1):
        page_url = _laodong_page_url(cat_url, page)
        log.info(f"  Page {page}: {page_url}")
        html = http.get_text(page_url)
        total["pages"] += 1
        if not html:
            consecutive_empty += 1
            if consecutive_empty >= 3:
                break
            continue
        urls = _extract_article_urls(html, page_url, LAODONG_ARTICLE_REGEX)
        new_urls = [u for u in urls if u not in seen_on_cat]
        if not new_urls:
            consecutive_empty += 1
            if consecutive_empty >= 3:
                break
            continue
        consecutive_empty = 0
        seen_on_cat.update(new_urls)
        page_stats = {"added": 0, "dup": 0, "invalid": 0, "fetch_failed": 0}
        for url in new_urls:
            r = _process_article_url(url, http, store)
            if r == "added":
                page_stats["added"] += 1
                total["added"] += 1
            elif r.startswith("invalid"):
                page_stats["invalid"] += 1
                total["invalid"] += 1
            elif r == "fetch_failed":
                page_stats["fetch_failed"] += 1
                total["fetch_failed"] += 1
            else:
                page_stats["dup"] += 1
                total["dup"] += 1
        log.info(f"    page stats: {page_stats}")

log.info("=" * 60)
log.info(f"FINAL: {total}")
log.info(f"Total in store: {store.count():,}")
log.info(f"Output: {JSONL_PATH}")
log.info("=" * 60)


## Cell 4 — ZNews HTML crawler

In [ ]:
# ====================================================================
# CELL 4 — ZNEWS HTML CRAWLER
# ====================================================================

ZNEWS_CATEGORIES = [
    "https://znews.vn/xuat-ban.html",
    "https://znews.vn/kinh-doanh-tai-chinh.html",
    "https://znews.vn/suc-khoe.html",
    "https://znews.vn/the-thao.html",
    "https://znews.vn/doi-song.html",
    "https://znews.vn/cong-nghe.html",
    "https://znews.vn/giai-tri.html",
]

START_DATE = "2026-01-18"
MAX_PAGES_PER_CATEGORY = 200
JSONL_PATH = "data/znews_articles.jsonl"

ZNEWS_ARTICLE_REGEX = re.compile(r"/[\w-]+/post\d+\.html")


def _znews_page_url(category_url, page):
    if page == 1:
        return category_url
    base = category_url.rsplit(".html", 1)[0]
    return f"{base}-trang{page}.html"


# -------- MAIN --------
http = HttpClient(min_delay_per_domain=0.8)
store = JsonlStore(JSONL_PATH)
start_local = iso_to_local_date(to_iso_utc(START_DATE + "T00:00:00"))

total = {"added": 0, "dup": 0, "invalid": 0, "fetch_failed": 0, "pages": 0}

for cat_url in ZNEWS_CATEGORIES:
    log.info(f">>> Category: {cat_url}")
    consecutive_empty = 0
    seen_on_cat = set()
    for page in range(1, MAX_PAGES_PER_CATEGORY + 1):
        page_url = _znews_page_url(cat_url, page)
        log.info(f"  Page {page}: {page_url}")
        html = http.get_text(page_url)
        total["pages"] += 1
        if not html:
            consecutive_empty += 1
            if consecutive_empty >= 3:
                break
            continue
        urls = _extract_article_urls(html, page_url, ZNEWS_ARTICLE_REGEX)
        new_urls = [u for u in urls if u not in seen_on_cat]
        if not new_urls:
            consecutive_empty += 1
            if consecutive_empty >= 3:
                break
            continue
        consecutive_empty = 0
        seen_on_cat.update(new_urls)
        page_stats = {"added": 0, "dup": 0, "invalid": 0, "fetch_failed": 0}
        for url in new_urls:
            r = _process_article_url(url, http, store)
            if r == "added":
                page_stats["added"] += 1
                total["added"] += 1
            elif r.startswith("invalid"):
                page_stats["invalid"] += 1
                total["invalid"] += 1
            elif r == "fetch_failed":
                page_stats["fetch_failed"] += 1
                total["fetch_failed"] += 1
            else:
                page_stats["dup"] += 1
                total["dup"] += 1
        log.info(f"    page stats: {page_stats}")

log.info("=" * 60)
log.info(f"FINAL: {total}")
log.info(f"Total in store: {store.count():,}")
log.info(f"Output: {JSONL_PATH}")
log.info("=" * 60)


## Cell 5 — 24h.com.vn RSS crawler

In [ ]:
# ====================================================================
# CELL 5 — 24H.COM.VN RSS CRAWLER
# ====================================================================

FEEDS_24H = [
    ("https://cdn.24h.com.vn/upload/rss/trangchu24h.rss", "trang-chu"),
    ("https://cdn.24h.com.vn/upload/rss/tintuctrongngay.rss", "tin-tuc-trong-ngay"),
    ("https://cdn.24h.com.vn/upload/rss/bongda.rss", "bong-da"),
    ("https://cdn.24h.com.vn/upload/rss/asiancup2019.rss", "the-thao"),
    ("https://cdn.24h.com.vn/upload/rss/thoitrang.rss", "thoi-trang"),
    ("https://cdn.24h.com.vn/upload/rss/thoitranghitech.rss", "hi-tech"),
    ("https://cdn.24h.com.vn/upload/rss/taichinhbatdongsan.rss", "tai-chinh-bat-dong-san"),
    ("https://cdn.24h.com.vn/upload/rss/phim.rss", "phim"),
    ("https://cdn.24h.com.vn/upload/rss/giaoducduhoc.rss", "giao-duc-du-hoc"),
    ("https://cdn.24h.com.vn/upload/rss/bantrecuocsong.rss", "ban-tre-cuoc-song"),
    ("https://cdn.24h.com.vn/upload/rss/thethao.rss", "the-thao"),
]

START_DATE = "2026-01-18"
ENRICH_FULL_CONTENT = True
JSONL_PATH = "data/24h_articles.jsonl"


# -------- MAIN --------
http = HttpClient(min_delay_per_domain=0.8)
store = JsonlStore(JSONL_PATH)
start_local = iso_to_local_date(to_iso_utc(START_DATE + "T00:00:00"))

total = {"fetched": 0, "added": 0, "dup": 0, "old": 0, "invalid": 0}

for feed_url, default_category in FEEDS_24H:
    log.info(f">>> Feed: {feed_url} (category: {default_category})")
    feed = feedparser.parse(feed_url)
    if not feed.entries:
        log.warning(f"  No entries")
        continue
    log.info(f"  {len(feed.entries)} entries")
    feed_stats = {"added": 0, "dup": 0, "old": 0, "invalid": 0}
    for entry in feed.entries:
        total["fetched"] += 1
        url = normalize_url(entry.get("link", ""))
        if not url:
            continue
        if store.has_url(url):
            total["dup"] += 1
            feed_stats["dup"] += 1
            continue
        pub_iso = _parse_entry_date(entry)
        if pub_iso and start_local:
            pub_local = iso_to_local_date(pub_iso)
            if pub_local and pub_local < start_local:
                total["old"] += 1
                feed_stats["old"] += 1
                continue
        article = build_article_from_rss(entry, http, enrich=ENRICH_FULL_CONTENT)
        if article is None:
            total["invalid"] += 1
            feed_stats["invalid"] += 1
            continue
        # Override category_raw bằng default_category của 24h nếu extractor không lấy được
        if not article.category_raw or article.category_raw == "trang-chu":
            article.category_raw = default_category
            article.category_normalized = normalize_category(default_category)
        if store.append(article):
            total["added"] += 1
            feed_stats["added"] += 1
        else:
            total["dup"] += 1
            feed_stats["dup"] += 1
    log.info(f"  → feed result: {feed_stats}")

log.info("=" * 60)
log.info(f"FINAL: {total}")
log.info(f"Total in store: {store.count():,}")
log.info(f"Output: {JSONL_PATH}")
log.info("=" * 60)


## Cell 6 — Kiểm tra kết quả (chạy sau khi đã crawl xong)

In [ ]:
# ====================================================================
# CELL 6 — VERIFY OUTPUT
# Xem nhanh chất lượng dữ liệu đã crawl từ tất cả sources
# ====================================================================

import glob
from collections import Counter

JSONL_FILES = sorted(glob.glob("data/*.jsonl"))
print(f"JSONL files found: {len(JSONL_FILES)}")
for f in JSONL_FILES:
    print(f"  {f}  ({os.path.getsize(f)/1024/1024:.1f} MB)")
print()

all_records = []
for f in JSONL_FILES:
    with open(f, "r", encoding="utf-8") as fp:
        for line in fp:
            try:
                all_records.append(json.loads(line))
            except json.JSONDecodeError:
                continue

print(f"=" * 60)
print(f"Total records: {len(all_records):,}")
print(f"Unique IDs   : {len(set(r['id'] for r in all_records)):,}")
print()

print("By source:")
for src, c in Counter(r["source"] for r in all_records).most_common():
    print(f"  {src:<20} {c:>6,}")
print()

print("By source_type:")
for st, c in Counter(r["source_type"] for r in all_records).most_common():
    print(f"  {st:<20} {c:>6,}")
print()

print("By category_normalized:")
for cat, c in Counter(r["category_normalized"] for r in all_records).most_common(15):
    print(f"  {cat:<20} {c:>6,}")
print()

full = sum(1 for r in all_records if r["has_full_content"])
with_author = sum(1 for r in all_records if r["author"])
with_keywords = sum(1 for r in all_records if r["keywords"])
print(f"Full content (>=500 chars): {full:,} ({full/len(all_records)*100:.1f}%)")
print(f"With author              : {with_author:,} ({with_author/len(all_records)*100:.1f}%)")
print(f"With keywords            : {with_keywords:,} ({with_keywords/len(all_records)*100:.1f}%)")
print()

# Sample
print("Sample record:")
print(json.dumps(all_records[0], ensure_ascii=False, indent=2)[:1500])
